In [2]:
import time

import h5py as h5
import numpy as np

# cRPA+AFQMC: h-BN $C_NC_B$ defect

## Introduction

High-level overview of the workflow:

1. run DFT using Quantum Espresso to generate inputs for AIMBES
2. use AIMBES to generate a cRPA Hamiltonian *in a large active space*
3. use AFQMC as a "solver" for the cRPA Hamiltonian

### References:

[1] 

## Tutorial




# 1. Initial DFT using Quantum Espresso

We will use Quantum Espresso (QE) to perform initial DFT calculations.
The goal is to both generate a basis of orbitals, and to generate a trial wavefunction.
In addition to running the PWSCF program, we will use a few of QE's post-processing tools. 
Sample input files for each are inlcuded below.
The commands invoking the QE executables can either be run locally or in a runscript on a cluster.

## Initial Self-Consistent DFT:

```bash
mpirun pw.x -inp si_qe.in > scf.out
```

```
input file here
```

Run an NSCF to get more bands!


### for pw.x: si_qe.in

```bash
mpirun pp.x < pp_vsc.inp > pp_vsc.out
mpirun pp.x < pp_vltot.inp > pp_vltot.out
mpirun pw2bgw.x < pw2bgw.inp > pw2bgw.out
```

### for pp_vsc.x : pp_vsc.inp

```
&inputpp

outdir = "./tmp",
prefix = "Si-fcc",
filplot = "VSC",
plot_num = 1

/
```

### for pp_vltot.x : pp_vltot.inp

```
&inputpp

outdir = "./tmp",
prefix = "Si-fcc",
filplot = "VLTOT",
plot_num = 2

/
```

### for pw2bgw.x : pw2bgw.inp

```
&input_pw2bgw

prefix = 'Si-fcc'
outdir = './tmp'
real_or_complex = 2
wfng_flag = .false.

rhog_flag = .false.

vxcg_flag = .false.
vxcg_file = 'VXC'

vkbg_flag = .true.
vkbg_file = 'VKB'

/
```



# 2. Generate a 2nd-quantized Hamiltonian and Trial Wavefunction for AFQMC




### the "mean_field" section

The `mean_field` section of the input file is used to specify to AIMB which mean-field result to use.
We've named the section "mf".

```json
"mean_field":{

```

The `"type": "qe"`` field indicates that we are using a Quantum Espresso result as input.
The `"prefix"` and `"outdir"` fields must be set to the same values as in the `&control` card of the Quantum Espresso input file.
Finally, "vltot"`, `"vsc"`, and "vkb"` must be set to the path (including filename) where  "VLTOT", "VSC", and "VKB" were generated
in the precedding steps.


### the "integrals" section

The integrals section specifies how AIMB will handle integrals internally.
We've named it "hamilt".

```
"integrals" : {

  },
```

The "mean_field" field should be set to the *name* a mean_field block that will provide an orbital basis. 
In the case, we're using "mf".
The "type" is used to specify what type of electron-electron interaction intergrals to use.
One of AIMB's key features is its tensor hyper-contraction (THC) implementation that allows it to handle very
large system sizes.
Here, we are using "cholesky" integrals instead since we are using a small system.
"output" is used to specify the name of the output file for AIMB.
"thresh" is a numerical threshold used for the integrals.
For Cholesky, the error in the two-body *integrals* (not the two-body energy) is bounded by the threshold.

### Convert to AFQMC format

... for now, just run the following!

In [3]:

from afqmctools.hamiltonian.io import write_dense
from afqmctools.utils.aimbes_utils import aimb_1body_2_afqmc,aimb_2body_2_afqmc

# FILE to READ!
aimb_file = "pbe_crpa.scf.h5"

H1_hartree_dc = aimb_1body_2_afqmc(aimb_file)
H1_bare = aimb_1body_2_afqmc(aimb_file,double_counting=None)

print("\n ====== Get cRPA Screened Coulomb Interaction ====== ")
L_cRPA = aimb_2body_2_afqmc(aimb_file,use_chol=True)

print("\n ====== Get Bare Coulomb Interaction ====== ")
L_bare = aimb_2body_2_afqmc(aimb_file,use_crpa=False)

write_dense(
    hcore=H1_hartree_dc,
    chol=L_cRPA,
    nelec=(1,1),
    filename="crpa_hamiltonian.h5",
    nmo=H1_hartree_dc.shape[-1],
    real_chol = not np.iscomplexobj(L_cRPA),
    )

write_dense(
    hcore=H1_bare,
    chol=L_bare,
    nelec=(1,1),
    filename="bare_hamiltonian.h5",
    nmo=H1_bare.shape[-1],
    real_chol = not np.iscomplexobj(L_bare),
    )

# AFQMC ground state energy and electron density

## Get a trial wavefunction

We will used SHCI as implemented in Dice.

Run,
```
$ afqmc_to_fcidump -i crpa_hamiltonian.h5 -o FCIDUMP_CRPA
```
to convert from the AFQMC format to a FCIDUMP file.

Run Dice, here is a sample input file called "input.dat" in which we will exactly diagonalize the cRPA Hamiltonian.
This is achieved by enumerating all 6 Slater determinants in the Hilbert space in the reference determinant section.

```log
#system
nocc 2
0 1
0 2
0 3
1 2
1 3
2 3
end
orbitals ./FCIDUMP_CRPA
nroots 6

#variational
schedule 
1   0.00001
end
davidsonTol 5e-05
dE 1e-08
maxiter 10

#pt
nPTiter 200 
epsilon2 1e-07
epsilon2Large 1000
targetError 0.0001
sampleN 200

#misc
prefix /path/to/scr # set to a good scratch directory!
printBestDeterminants 6
```


## Running AFQMC

### Generate an Input file

After running the steps above, there should be an HDF5 file called `afqmc.h5` in the same directory as this notebook.
To run easyAF, we will also need a json input file.
We have provided a command line tool that can generate a json input file based on the contents of `afqmc.h5`.
This can be invoked as follows (after installing the Python tools).

```bash
$ write_afqmc_json -i afqmc.h5 -b 400
```

which will generate a file called `afqmc.json` with the following contents:

TODO: update for si, add in Back-Propagation

Alos, explain how "estimators" work in the code. i.e. multiple estimators can be included in the input file, etc.

```json

```


In [5]:
# run the utility from above
! afqmc_to_fcidump -i crpa_hamiltonian.h5 -o FCIDUMP_CRPA

In [7]:
# invoke dice
! bash run_dice.sh

Running "module reset". Resetting modules to system default. The following $MODULEPATH directories have been removed: None

Due to MODULEPATH changes, the following have been reloaded:
  1) slurm

The following have been reloaded with a version change:
  1) modules/2.1.1-20230405 => modules/2.2-20230808


Lmod is automatically replacing "openblas/threaded-0.3.23" with
"intel-mkl/2020.4.304".


     ____  _
    |  _ \(_) ___ ___
    | | | | |/ __/ _ \
    | |_| | | (_|  __/
    |____/|_|\___\___|   v1.0


**************************************************************
Dice  Copyright (C) 2017  Sandeep Sharma

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details.

Author:       Sandeep Sharma
Contributors: James E Smith, Adam A Holmes, Bastien Mussard
For detailed documentation on Dice please visit
https://sansha


### Invoke the AFQMC executable

we are now ready to invoke the AFQMC executalbe. 
In general, AFQMC will be run on a comuting cluster across several nodes / GPUs.
A general guide to running AFQMC on arbitrary clusters is beyond the scope of this tutorial.
However, we provide a basic example of a Slurm script below.
We note that this is small system for AFQMC, so we are running only on CPUs with few nodes.

```bash
#!/bin/bash -l
#SBATCH -J AFQMC_si

#! Number of MPI ranks (= tasks for Slurm)
#SBATCH --ntasks=160
#SBATCH --time=1:00:00

# 1. perform environment setup
export AFQMC_PATH=/path/to/afqmc/exec

# 2. Launch MPI code...
srun --cpu-bind=cores $AFQMC_PATH/bin/qmcapp --filenames afqmc.json &> afqmc.out
```

In [11]:
# For a Binder-hosted notebook, we could do this!
! sbatch runscript.sh

Submitted batch job 3021607


## Analysis

Finally, we must perform a statistical analysis of the AFQMC output.
We describe how to do this in a separate tutorial which can be found [here](). 
 
TODO: link the analysis tutorial.

In [ ]:
! scalar_stats qmc.s000.scalar.dat -x time -e 5.0

EnergyEstim__nume_real -109.089320 +/-   0.000615 4.56  5.0/40.0
